<a href="https://colab.research.google.com/github/profbnk/AI/blob/main/Study_planner_by_Laya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📚 Study Planner Documentation (Powered by Laya)

This application uses the **Laya Router** to dynamically recommend academic actions for a student based on their current situation. Unlike rigid rule-based hardcoding, Laya uses semantic understanding to matching the student's status with descriptive decision criteria.

### ⚙️ How it Works
1. **State Aggregation**: The system collects parameters from user-interactive slider widgets (subject, days left, free time, confidence, energy).
2. **Context Compilation**: This data is compiled into a simple, structured textual representation (`state_text`).
3. **Laya Prediction**: Laya's `Router` receives the state profile and matches it against explicit semantic criteria for each available outcome.
4. **Action & Output**: The application retrieves the corresponding advice string mapping from the determined classification.

---

### 🧠 Decision Matrix & Criteria

Below is the semantic logic defining each of Laya's decisions:

| Recommendation Choice | Intent | Operational Criteria Conditions |
| :--- | :--- | :--- |
| **Study Foundations** | Deep-focus on building core concepts. | Energy is decent ($\\ge$ 4), exam is not urgent ($>$ 7 days left), and confidence is low ($\\le$ 5). |
| **Practice Past Papers** | Application under exam-style timers. | Exam is coming up (8 to 30 days left), confidence is moderate ($\\ge$ 5), and energy levels are high ($\\ge$ 5). |
| **Simulate Mock Test** | Full assessment under strict constraints. | Exam is near (between 3 and 7 days left) and energy is healthy ($\\ge$ 6). |
| **Quick Formula Review** | Low-stress refresher/cramming. | Exam is extremely imminent ($\\le$ 2 days left) and student has energy ($\\ge$ 4). |
| **Recharge and Rest** | Prevent burnout and refresh brain. | Energy is critically low ($\\le$ 3) OR free time is extremely restricted ($\\le$ 1 hour) regardless of days left. |

In [ ]:
!pip install laya

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------
# 1.  LAYA CODE HERE
# ---------------------------------------------------
from laya import Router

router = Router()  # loads once; reused on every button click

def laya_decide(state, question):
    # Laya reads the state as plain text
    state_text = (
        f"Current State:\n"
        f"- Subject: {state['weak_subject']}\n"
        f"- Days until exam: {state['days_left']}\n"
        f"- Hours of free time today: {state['hours_free']}\n"
        f"- Student's confidence: {state['weak_confidence']}\n"
        f"- Student's energy level: {state['energy']}"
    )
    questions = {
        "action": {
            "type": "choice",
            "instructions": question,
            "criteria": {
                "Study Foundations": "Choose this ONLY if energy is >= 4, AND days left is strictly greater than 7 (days > 7), AND confidence is low (<= 5). This is for deep, early-stage studying.",
                "Practice Past Papers": "Choose this ONLY if days left is between 8 and 30, AND confidence is medium-to-high (>= 5), AND energy is high (>= 5).",
                "Simulate Mock Test": "Choose this ONLY if days left is close (between 3 and 7 days left) AND energy is healthy (>= 6).",
                "Quick Formula Review": "Choose this ONLY if the exam is extremely close (under 3 days left, i.e., days <= 2) AND energy is >= 4.",
                "Recharge and Rest": "Choose this immediately if energy is critically low (energy <= 3), OR if free time is extremely restricted (hours_free <= 1), regardless of days left."
            }
        }
    }
    result = router.predict(state_text, questions)
    # Diagnostic logging to see scores for each option
    print("\n--- [Laya Debug Diagnostics] ---")
    print("State text input:\n", state_text)
    print("Raw decision dictionary:\n", result)
    print("--------------------------------\n")
    return result["answers"]["action"]["choice"]

# Mapping the new distinct choices to the user's advice dictionary
ACTIONS = {
    "Study Foundations": "Deep-focus on your weakest subject (45 min blocks).",
    "Practice Past Papers": "Solve previous-year questions under a time limit.",
    "Simulate Mock Test": "Attempt a full mock test and review mistakes.",
    "Quick Formula Review": "Quick notes and formulas revision. Keep it relaxed.",
    "Recharge and Rest": "Rest, walk, sleep. A fresh brain learns faster.",
}

# ---------------------------------------------------
# 2. SIMPLE UI (sliders + button, focused solely on Laya)
# ---------------------------------------------------
subject = widgets.Text(value="Data Structures", description="Weak subject")
days = widgets.IntSlider(value=10, min=0, max=60, description="Days left")
hours = widgets.FloatSlider(value=3, min=0, max=12, step=0.5, description="Free hours")
conf = widgets.IntSlider(value=4, min=1, max=10, description="Confidence")
energy = widgets.IntSlider(value=6, min=1, max=10, description="Energy")
button = widgets.Button(description="Ask Laya", button_style="success")
out = widgets.Output()

def on_click(_):
    state = {
        "weak_subject": subject.value,
        "days_left": days.value,
        "hours_free": hours.value,
        "weak_confidence": conf.value,
        "energy": energy.value,
    }

    with out:
        clear_output()
        try:
            decision = laya_decide(state, "What should the student do next?")
            print("RECOMMENDATION:", decision)
            print("ADVICE        :", ACTIONS.get(decision, "No advice mapped for this decision."))
        except Exception as e:
            print("An error occurred:", e)

button.on_click(on_click)
display(widgets.VBox([subject, days, hours, conf, energy, button, out]))